# Practice: pandas and scikit-learn
*AI from scratch, toolkit 2* · lesson: https://ihor.sh/ai/pandas-and-sklearn.html

Five weeks of NYC subway elevator and escalator outages: load, look, filter, group, merge, then a pipeline that
predicts which outages will last more than 4 hours, scored with a time split and tuned with a grid search.
Replace each `...` under `# YOUR CODE HERE`, run the cell, then the check under it; `hint(n)` and `answer(n)` are
there when you're stuck.

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['(df["kind"] == "EL") is a column of True/False.', 'n_el = (df["kind"] == "EL").sum()'], 2: ['Three conditions: (f["kind"] == "EL"), (f["ada"] == 1), (f["hours"] > 4).', 'long_ada = f[(f["kind"] == "EL") & (f["ada"] == 1) & (f["hours"] > 4)]'], 3: ['f["weekday"] = f["start"].dt.day_name()', 'busiest_day = f["weekday"].value_counts().idxmax()'], 4: ['f.groupby("reason")["hours"] then a summary.', 'med = f.groupby("reason")["hours"].median()'], 5: ['Group by "station", then .agg(...) with the two named summaries.', 'Finish with .sort_values("outages", ascending=False)'], 6: ['print(boro): one station is listed twice, so its outages each match two rows.', 'Remove the repeat first: boro.drop_duplicates(), then merge with validate="many_to_one" to make pandas check.'], 7: ['scores = cross_val_score(model, X, y, cv=TimeSeriesSplit(n_splits=5), scoring="roc_auc"), then their mean.'], 8: ['grid = GridSearchCV(model, {"logisticregression__C": [0.01, 0.1, 1, 10, 100]}, cv=TimeSeriesSplit(n_splits=5), scoring="roc_auc")', 'grid.fit(X, y), then best_C = grid.best_params_["logisticregression__C"]']}
_ANSWERS = {1: 'bl9lbCA9IChkZlsia2luZCJdID09ICJFTCIpLnN1bSgp', 2: 'bG9uZ19hZGEgPSBmWyhmWyJraW5kIl0gPT0gIkVMIikgJiAoZlsiYWRhIl0gPT0gMSkgJiAoZlsiaG91cnMiXSA+IDQpXQ==', 3: 'Zlsid2Vla2RheSJdID0gZlsic3RhcnQiXS5kdC5kYXlfbmFtZSgpCmJ1c2llc3RfZGF5ID0gZlsid2Vla2RheSJdLnZhbHVlX2NvdW50cygpLmlkeG1heCgp', 4: 'bWVkID0gZi5ncm91cGJ5KCJyZWFzb24iKVsiaG91cnMiXS5tZWRpYW4oKQ==', 5: 'Ynlfc3RhdGlvbiA9IGYuZ3JvdXBieSgic3RhdGlvbiIpLmFnZyhvdXRhZ2VzPSgiaG91cnMiLCAic2l6ZSIpLCBtZWRpYW5faG91cnM9KCJob3VycyIsICJtZWRpYW4iKSkuc29ydF92YWx1ZXMoIm91dGFnZXMiLCBhc2NlbmRpbmc9RmFsc2Up', 6: 'Zml4ZWQgPSBmLm1lcmdlKGJvcm8uZHJvcF9kdXBsaWNhdGVzKCksIG9uPSJzdGF0aW9uIiwgaG93PSJsZWZ0IiwgdmFsaWRhdGU9Im1hbnlfdG9fb25lIik=', 7: 'YXVjID0gY3Jvc3NfdmFsX3Njb3JlKG1vZGVsLCBYLCB5LCBjdj1UaW1lU2VyaWVzU3BsaXQobl9zcGxpdHM9NSksIHNjb3Jpbmc9InJvY19hdWMiKS5tZWFuKCk=', 8: 'Z3JpZCA9IEdyaWRTZWFyY2hDVihtb2RlbCwgeyJsb2dpc3RpY3JlZ3Jlc3Npb25fX0MiOiBbMC4wMSwgMC4xLCAxLCAxMCwgMTAwXX0sIGN2PVRpbWVTZXJpZXNTcGxpdChuX3NwbGl0cz01KSwgc2NvcmluZz0icm9jX2F1YyIpCmdyaWQuZml0KFgsIHkpCmJlc3RfQyA9IGdyaWQuYmVzdF9wYXJhbXNfWyJsb2dpc3RpY3JlZ3Jlc3Npb25fX0MiXQ=='}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import numpy as np
import pandas as pd

DATA = "https://raw.githubusercontent.com/ikruchynskyi/ihor.sh/main/ai/data/"
df = pd.read_csv(DATA + "outages.csv", parse_dates=["start", "estimated_end", "last_seen"])
print(df.shape)
df.info()
df.head()

## 1. Count something
`kind` is `"EL"` for elevators and `"ES"` for escalators. Set `n_el` to the number of elevator outages: a
comparison gives a True/False column, and `.sum()` counts the Trues.

In [ ]:
n_el = None
# YOUR CODE HERE (stuck? hint(1), then answer(1))
...
print(n_el)

In [ ]:
check(1, lambda: n_el == df["kind"].value_counts()["EL"], "the number of rows with kind == 'EL' (value_counts agrees)")

## 2. How long each outage lasted
The finished outages, with their length in hours (given), then your filter: set `long_ada` to the elevator
outages (`kind == "EL"`) at accessible stations (`ada == 1`) that lasted more than 4 hours. Each condition in
brackets, joined with `&`.

In [ ]:
f = df[df["finished"]].copy()
f["hours"] = (f["last_seen"] - f["start"]).dt.total_seconds() / 3600
long_ada = None
# YOUR CODE HERE (stuck? hint(2), then answer(2))
...
print(len(long_ada) if long_ada is not None else None, "long elevator outages at accessible stations")

In [ ]:
check(2, lambda: len(long_ada) == ((f["kind"] == "EL") & (f["ada"] == 1) & (f["hours"] > 4)).sum() and long_ada["hours"].min() > 4,
      "rows where all three hold: (cond) & (cond) & (cond)")

## 3. A column from a date
Add a column `weekday` to `f` with the day's name of each outage's start. Dates have a `.dt` part, like text has
`.str`; the name of the day is `f["start"].dt.day_name()`. Then `busiest_day` is the most common one
(`.value_counts().idxmax()` gives the most frequent value).

In [ ]:
busiest_day = None
# YOUR CODE HERE (stuck? hint(3), then answer(3))
...
print(busiest_day)

In [ ]:
check(3, lambda: "weekday" in f and busiest_day == f["start"].dt.day_name().value_counts().idxmax(),
      "f['weekday'] holds day names; the most common one is the busiest day")

## 4. Group and summarize
Set `med` to the median hours per reason: group by `"reason"`, take the `"hours"` column, then `.median()`.
**Before you run it, predict:** will "Cleaning" or "Maintenance" have the longer median?

In [ ]:
med = None
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...
print(med.sort_values().round(2) if med is not None else None)

In [ ]:
check(4, lambda: np.isclose(med["Cleaning"], f.loc[f["reason"] == "Cleaning", "hours"].median()) and med["Maintenance"] > med["Cleaning"],
      "one median per reason; cleaning is quick (under an hour), maintenance takes hours")

## 5. Several summaries per station
Set `by_station` to a table with one row per station and two columns: `outages` (how many) and `median_hours`.
Use `.agg` with names you choose, as in the lesson: `agg(outages=("hours", "size"), median_hours=("hours", "median"))`.
Then sort it by `outages`, most first.

In [ ]:
by_station = None
# YOUR CODE HERE (stuck? hint(5), then answer(5))
...
by_station.head() if by_station is not None else None

In [ ]:
check(5, lambda: list(by_station.columns) == ["outages", "median_hours"] and by_station["outages"].iloc[0] == f["station"].value_counts().iloc[0]
      and by_station["outages"].sum() == len(f),
      "columns outages and median_hours, sorted with the busiest station first; the counts add up to every finished outage")

## 6. Find the bug: a merge that multiplies
Someone made a table of station boroughs, then merged it onto the outages. Run it: the row count goes *up*.
Find why, then set `fixed` to the merge done right, with the same number of rows as `f`.

In [ ]:
boro = pd.DataFrame({"station": ["34 St-Herald Sq", "34 St-Herald Sq", "72 St", "Grand Central-42 St"],
                     "borough": ["Manhattan", "Manhattan", "Manhattan", "Manhattan"]})
merged = f.merge(boro, on="station", how="left")
print("rows before:", len(f), " after:", len(merged))
fixed = None
# YOUR CODE HERE (stuck? hint(6), then answer(6))
...

In [ ]:
check(6, lambda: len(fixed) == len(f) and "borough" in fixed, "as many rows as f: each outage matches at most one borough row")

## 7. A pipeline, scored on the future
Predict whether an outage lasts more than 4 hours. The features and the preprocessing are given. Set `auc` to the
mean AUC of `model` under `cross_val_score` with `cv=TimeSeriesSplit(n_splits=5)` and `scoring="roc_auc"`.

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, TimeSeriesSplit, GridSearchCV

g = f.sort_values("start").copy()
g["hour"] = g["start"].dt.hour
g["n_lines"] = g["lines"].str.count("/") + 1
X = g[["kind", "reason", "ada", "hour", "n_lines"]]
y = (g["hours"] > 4).astype(int)
prep = make_column_transformer((OneHotEncoder(handle_unknown="ignore"), ["kind", "reason"]), (StandardScaler(), ["ada", "hour", "n_lines"]))
model = make_pipeline(prep, LogisticRegression(max_iter=2000))
print("base rate of long outages:", y.mean().round(3))
auc = None
# YOUR CODE HERE (stuck? hint(7), then answer(7))
...
print("time-split AUC:", auc)

In [ ]:
check(7, lambda: 0.8 < auc < 0.97, "about 0.9: the reason for an outage says a lot about how long it lasts")

## 8. Tune it
Search the penalty strength `C` over `[0.01, 0.1, 1, 10, 100]` with `GridSearchCV`, the same time split and
scoring. In a pipeline, the setting is named `"logisticregression__C"` (step name, two underscores, setting).
Set `best_C` to the winner (`grid.best_params_["logisticregression__C"]`).

In [ ]:
best_C = None
# YOUR CODE HERE (stuck? hint(8), then answer(8))
...
print("best C:", best_C)

In [ ]:
check(8, lambda: best_C in [0.01, 0.1, 1, 10, 100] and best_C == GridSearchCV(model, {"logisticregression__C": [0.01, 0.1, 1, 10, 100]},
      cv=TimeSeriesSplit(n_splits=5), scoring="roc_auc").fit(X, y).best_params_["logisticregression__C"],
      "the C with the best mean time-split AUC")

The grid's best score is a little optimistic: it was used to choose. Keep the newest outages untouched as a
final test when the archive grows.

## Next
Back to the path: https://ihor.sh/ai/ · Kaggle Learn's Pandas course: https://www.kaggle.com/learn/pandas